# Faster R-CNN — final seeds, một seed hoặc hàng đợi

Giữ winner sau 8 trial + 5 ablation. Hai phiên Kaggle có thể chạy song song:
phiên A `SEED = [7, 21]`, phiên B `SEED = [123, 2026]`.
Trong mỗi phiên, seed chạy **nối tiếp**, mỗi seed dùng hai T4 và có W&B run + ZIP/checksum riêng.
Mặc định dùng lại seed42 `ahgf5kge`; có thể chọn 42 để chạy một lượt lặp được ghi rõ.
Không tính hai run của seed42 thành hai seed độc lập, không chọn run theo điểm.

T4×2, Internet ON, Secret WANDB_API_KEY. Attach cùng cache và Output tuning cuối ZIP e2d80e82 + checksum.
Save & Run All thực sự train. Giữ source a99d294, cache/runtime và hyperparameters đã khóa.
Notebook lấy recipe từ reference bundle; YAML mặc định hiện tại là bản portable của cùng winner.
Không load checkpoint winner hoặc resume training. Tổng thời gian hàng đợi phải nằm trong giới hạn phiên Kaggle.


In [ ]:
SEED = [7, 21]  # Phiên A; phiên B: [123, 2026]. Một seed: 21; chạy lại seed42: [42].
REFERENCE_BUNDLE = ""  # Tự tìm bundle e2d80e82; hoặc Copy file path nếu nhiều bản.
RESTORE_OUTPUT = ""  # Một seed: đường dẫn ZIP; nhiều seed: {7: "...zip", 21: "...zip"}.
START_TRAIN = True

def normalize_seeds(value):
    if type(value) is int:
        seeds = [value]
    else:
        assert isinstance(value, (list, tuple)), "Dùng int hoặc list [21, 7]; không dùng set {21, 7} vì không giữ thứ tự"
        seeds = list(value)
    assert 1 <= len(seeds) <= 5, "Mỗi phiên cần 1–5 seed"
    assert all(type(seed) is int and seed in {42, 7, 21, 123, 2026} for seed in seeds), "Chỉ nhận 42, 7, 21, 123, 2026 (24 không phải 42)"
    assert len(seeds) == len(set(seeds)), "Không lặp seed trong cùng hàng đợi"
    return seeds

SEEDS = normalize_seeds(SEED)
print("Thứ tự chạy:", SEEDS)


In [ ]:
import os, sys, subprocess, json, re
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_REF = "a99d2946b55305cea8c6049bd6f581ab709f876a"  # SHA đầy đủ của commit có tích hợp Sweeps, đã push; không dùng tên branch.
assert re.fullmatch(r"[0-9a-f]{40}", REPO_REF), "Điền SHA đã commit/push trước khi chạy."
assert Path("/kaggle/working").exists(), "Notebook này dành cho Kaggle."
REPO = Path("/kaggle/working/VN-Traffic-Sign-Detection")

def run(command, **kwargs):
    return subprocess.run([str(x) for x in command], check=True, **kwargs)

if not (REPO / ".git").exists():
    run(["git", "clone", "--no-checkout", REPO_URL, REPO])
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True)
    assert not dirty.strip(), "Repo có tracked changes; giữ lại trước khi checkout."
run(["git", "-C", REPO, "fetch", "origin", REPO_REF])
run(["git", "-C", REPO, "checkout", "--detach", REPO_REF])
os.chdir(REPO)
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[detect]", "wandb==0.31.0"])
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import torch, torchvision, wandb
from torchvision.ops import nms
from kaggle_secrets import UserSecretsClient
from vn_tsd.utils.io import load_yaml, save_yaml

shared = load_yaml(REPO / "configs/shared.yaml")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_BASE_URL"] = shared["project"]["wandb_base_url"]
os.environ["WANDB_MODE"] = "online"
print("torch", torch.__version__, "torchvision", torchvision.__version__, "wandb", wandb.__version__)
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Cần đúng hai GPU T4."
for index in range(2):
    device = f"cuda:{index}"
    result = nms(torch.tensor([[0., 0., 10., 10.]], device=device), torch.tensor([.9], device=device), .5)
    assert result.tolist() == [0]
    print(index, torch.cuda.get_device_name(index))
for key in ("WANDB_SWEEP_ID", "WANDB_RUN_ID", "WANDB_CONFIG_PATHS", "WANDB_RESUME"):
    os.environ.pop(key, None)  # Notebook này không thuộc campaign tuning cũ.


## Kiểm reference và khóa cấu hình — chưa training

Kiểm SHA256/CRC của bundle, winner/config/source/runtime/cache như ablation.
Giữ nguyên recipewinner, chỉ thay seed và project tracking BTL. Không dùng YAMLbaseline để train.
Các JSON/YAML tham chiếu được đọc trực tiếp; không giải nén hoặc load checkpointwinner.


In [ ]:
import copy, hashlib, math, stat, uuid
from zipfile import ZipFile, ZIP_STORED
from vn_tsd.pipelines.faster_rcnn.sweep import source_identity, runtime_identity
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash, file_hash, load_cache
from vn_tsd.pipelines.faster_rcnn.config import validate_config
from vn_tsd.pipelines.faster_rcnn.artifacts import write_json

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working")
REFERENCE_NAME = "frcnn_tune_s42_es10_bayes8_sandbox_v1-artifacts-e2d80e82.zip"
REFERENCE_SHA256 = "3c817755f338f50e5300fc5e9b5b2b3604969a2c6bf58a384f714036f46f394f"
WINNER_ID = "ahgf5kge"
REFERENCE_CAMPAIGN = "frcnn_tune_s42_es10_bayes8_sandbox_v1"
EXPECTED_WINNER = {"train.lr": 0.009606472124890571,
                   "train.weight_decay": 1.4756984677083597e-05,
                   "model.head_dropout": 0.4, "online_preprocess.jitter": 0.1}

def read_reference(bundle):
    bundle = Path(bundle)
    checksum = bundle.with_suffix(".zip.sha256")
    assert checksum.is_file(), f"Thiếu checksum: {checksum}"
    actual = file_hash(bundle)
    assert actual == REFERENCE_SHA256 == checksum.read_text().split()[0], "Sai bundle/checksum tham chiếu"
    root = REFERENCE_CAMPAIGN + "/"
    with ZipFile(bundle) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist())), "Duplicate ZIP entries"
        assert archive.testzip() is None, "Reference ZIP CRC error"
        read = lambda name: json.loads(archive.read(root + name))
        frozen = read("campaign.json")
        base = __import__("yaml").safe_load(archive.read(root + "base_config.yaml"))
        search = __import__("yaml").safe_load(archive.read(root + "sweep.yaml"))
        assert content_hash(base) == frozen["base_config_sha256"]
        assert content_hash(search) == frozen["sweep_sha256"] and search["run_cap"] == 8
        assert read("sweep_id.json")["sweep_id"] == "upxyjxyl"
        prefix = "trials/" + WINNER_ID + "/"
        cfg = __import__("yaml").safe_load(archive.read(root + prefix + "resolved_config.yaml"))
        metrics = read(prefix + "metrics.json")
        trial = read(prefix + "sweep_trial.json")
        dataset = read(prefix + "dataset_identity.json")
        assert metrics["status"] == "completed" and metrics["best_epoch"] == 49
        assert metrics["best_val"]["mAP50-95"] == 0.4905282939017382
        assert content_hash(cfg) == metrics["config_sha256"] == trial["config_sha256"]
        assert content_hash(dataset) == trial["dataset_sha256"] == frozen["dataset_sha256"]
        assert trial["parameters"] == EXPECTED_WINNER
        assert frozen["nproc"] == 2 and frozen["test_policy"] == "shared"
        for key, value in EXPECTED_WINNER.items():
            section, field = key.split(".")
            assert cfg[section][field] == value
        return cfg, base, frozen, metrics

def make_final(winner, seed, data_root):
    assert type(seed) is int and seed in {42, 7, 21, 123, 2026}, "Seed ngoài protocol đã chốt"
    cfg = copy.deepcopy(winner)
    cfg["seed"] = seed
    cfg["train"].pop("run_name", None)
    cfg["data"]["root"] = str(Path(data_root).resolve())
    cfg["project"]["wandb_project"] = "BTL"
    cfg = validate_config(cfg)
    assert cfg["model"]["pretrained"] and cfg["train"]["require_wandb"]
    assert cfg["train"]["epochs"] == 100 and cfg["train"]["warmup_epochs"] == 3
    assert cfg["train"]["early_stopping_patience"] == 10
    assert cfg["train"]["batch_size"] == 24 and cfg["eval"]["batch_size"] == 32
    assert cfg["eval"]["auto_eval_test"] and cfg["online_preprocess"]["enabled"]
    assert cfg["model"]["head"] == "custom_mlp" and cfg["model"]["head_hidden_dim"] == 512
    for key, value in EXPECTED_WINNER.items():
        section, field = key.split(".")
        assert cfg[section][field] == value
    return cfg

if REFERENCE_BUNDLE:
    bundle_path = Path(REFERENCE_BUNDLE)
else:
    candidates = sorted(INPUT_ROOT.rglob(REFERENCE_NAME))
    assert len(candidates) == 1, "Cần đúng 1 ZIP e2d80e82 + checksum; nếu nhiều thì điền REFERENCE_BUNDLE"
    bundle_path = candidates[0]
winner, baseline, frozen, winner_metrics = read_reference(bundle_path)
assert source_identity() == frozen["source_files"], "Source khác tuning; giữ pin a99d294"
assert runtime_identity() == frozen["runtime"], "Runtime đổi; gửi lỗi để review, không lách metadata"
DATA_ROOT = Path(load_yaml("configs/runtime/kaggle.yaml")["data_root_default"])
assert DATA_ROOT.is_dir(), f"Thiếu cache: {DATA_ROOT}"
# Cache được kiểm một lần trước khi bắt đầu hàng đợi; worker vẫn kiểm từng run.
_, dataset = load_cache(make_final(winner, SEEDS[0], DATA_ROOT))
assert content_hash(dataset) == frozen["dataset_sha256"], "Cache khác tuning"

# Khôi phục output của chính seed này, không khôi phục tuning campaign.
def restore_output(bundle, target, expected_request):
    bundle, target = Path(bundle), Path(target).resolve()
    assert not target.exists(), "Không ghi đè output đã có"
    assert file_hash(bundle) == bundle.with_suffix(".zip.sha256").read_text().split()[0]
    target.parent.mkdir(parents=True, exist_ok=True)
    staging = target.parent / (".restore_" + uuid.uuid4().hex)
    staging.mkdir()
    with ZipFile(bundle) as archive:
        assert len(archive.namelist()) == len(set(archive.namelist()))
        for entry in archive.infolist():
            dest = (staging / entry.filename).resolve()
            assert "\\" not in entry.filename and dest.is_relative_to(staging / target.name), "Unsafe path"
            assert not stat.S_ISLNK(entry.external_attr >> 16), "Symlink rejected"
        assert archive.testzip() is None
        archive.extractall(staging)
    restored = staging / target.name
    assert json.loads((restored / "request.json").read_text()) == expected_request, "Sai seed/config/source"
    restored.rename(target)

def normalize_restore(value, seeds):
    if not value:
        return {}
    if isinstance(value, str):
        assert len(seeds) == 1, "Nhiều seed: RESTORE_OUTPUT phải là dict {seed: đường_dẫn_zip}"
        return {seeds[0]: value}
    assert isinstance(value, dict), "RESTORE_OUTPUT phải là chuỗi hoặc dict"
    assert all(type(key) is int and key in seeds and isinstance(path, str) and path for key, path in value.items()), "Restore key/path không hợp lệ"
    return value

def prepare_final(seed, restore_bundle=""):
    cfg = make_final(winner, seed, DATA_ROOT)
    name = f"frcnn_final_custom_s{seed}_ep100_bs24_online_v1"
    work = OUTPUT_ROOT / "frcnn_final_seeds" / name
    request = dict(protocol_id="frcnn_final_v1", seed=seed, winner_run_id=WINNER_ID, reference_sha256=REFERENCE_SHA256,
                   reference_sweep_id="upxyjxyl", config=cfg, dataset_sha256=frozen["dataset_sha256"],
                   source_files=frozen["source_files"], runtime=frozen["runtime"],
                   initialization="fresh COCO pretrained; winner checkpoint not loaded",
                   selection="best validation mAP50-95; test report only")
    if seed == 42:
        request["repeat_of"] = WINNER_ID
        request["aggregation"] = "Repeat only; original seed42 remains the planned aggregate member"
    if restore_bundle:
        assert not work.exists(), "Output đã có; bỏ RESTORE_OUTPUT để dùng bản đã kiểm tại Working"
        restore_output(restore_bundle, work, request)
    if work.exists():
        assert json.loads((work / "request.json").read_text()) == request, "Output khác seed; không ghi đè"
    else:
        work.mkdir(parents=True)
        write_json(request, work / "request.json")
    print("Final seed:", seed, "Project: BTL; Output:", work)
    return dict(cfg=cfg, request=request, WORK=work, NAME=name, seed=seed)

restores = normalize_restore(RESTORE_OUTPUT, SEEDS)
JOBS = [prepare_final(seed, restores.get(seed, "")) for seed in SEEDS]
print("Đã kiểm reference/source/runtime/cache. Hàng đợi:", SEEDS)
print(json.dumps(JOBS[0]["cfg"], indent=2))


## Hàng đợi nối tiếp — mỗi seed một run và một ZIP/checksum

START_TRAIN=True đã đặt sẵn. Mỗi seed train mới từ COCO pretrained và export ngay sau run.
Completed output được kiểm rồi skip; failed/dở dừng cả hàng đợi để review, không tự retry.
Có thể chạy lại cell này để bỏ qua các run đã completed trong cùng Working.
Nếu runtime reset, attach output ZIP/checksum và khai báo RESTORE_OUTPUT theo seed cần khôi phục.


In [ ]:
from IPython.display import display, FileLink

def export_output(work):
    bundle = OUTPUT_ROOT / f"{work.name}-artifacts-{uuid.uuid4().hex[:8]}.zip"
    # W&B creates log/latest-run symlinks on Linux. Skip links instead of
    # dereferencing them; canonical regular files remain in the archive.
    skipped = sorted(p.relative_to(work).as_posix() for p in work.rglob("*") if p.is_symlink())
    files = sorted(p for p in work.rglob("*") if not p.is_symlink() and p.is_file()
                   and p.suffix.lower() not in {".zip", ".sha256"}
                   and p.name not in {".netrc", "_netrc", ".env"}
                   and p.relative_to(work).as_posix() != "export_receipt.json")
    with ZipFile(bundle, "x", compression=ZIP_STORED, allowZip64=True) as archive:
        for path in files:
            assert not path.is_symlink(), f"Output changed during export: {path.name}"
            archive.write(path, f"{work.name}/{path.relative_to(work).as_posix()}")
        archive.writestr(f"{work.name}/export_receipt.json", json.dumps(
            {"skipped_symlinks": skipped, "regular_files": len(files)}, indent=2))
    with ZipFile(bundle) as archive:
        assert archive.testzip() is None
    checksum = bundle.with_suffix(".zip.sha256")
    checksum.write_text(file_hash(bundle) + "  " + bundle.name + "\n", encoding="utf8")
    print("Tải ZIP + checksum ở Output gốc:", bundle, checksum, sep="\n")
    display(FileLink(str(bundle))); display(FileLink(str(checksum)))
    return bundle

def verify_completed(work, cfg, request):
    receipt = json.loads((work / "launch_status.json").read_text())
    assert receipt["status"] == "completed", "Run lỗi/dở: giữ output và review; không tự train lại"
    directory = work / "run"
    metrics = json.loads((directory / "metrics.json").read_text())
    saved = load_yaml(directory / "resolved_config.yaml")
    assert metrics["status"] == "completed" and metrics["config_sha256"] == content_hash(saved)
    expected = copy.deepcopy(cfg); expected["train"]["run_name"] = saved["train"]["run_name"]
    assert expected == saved
    assert content_hash(json.loads((directory / "dataset_identity.json").read_text())) == request["dataset_sha256"]
    assert file_hash(directory / "checkpoints/checkpoint_best.pt") == metrics["checkpoint_sha256"]
    tracking = json.loads((directory / "wandb_run.json").read_text())
    assert tracking["id"] == receipt["run_id"]
    score = metrics["best_val"]["mAP50-95"]
    assert isinstance(score, (int, float)) and math.isfinite(score)
    return metrics

# Worker được ghi ngoài repo pin; không sửa source training hoặc dùng checkpoint winner.
WORKER_CODE = r"""
import json, os, sys
from pathlib import Path
from vn_tsd.utils.io import load_yaml
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash, load_cache
from vn_tsd.pipelines.faster_rcnn.sweep import source_identity, runtime_identity
from vn_tsd.pipelines.faster_rcnn.train import FasterRCNNPipeline
work = Path(sys.argv[1])
request = json.loads((work / 'request.json').read_text())
cfg = load_yaml(work / 'train_config.yaml')
expected = request['config']
comparison = json.loads(json.dumps(cfg)); comparison['train'].pop('run_name', None)
assert comparison == expected
assert source_identity() == request['source_files'] and runtime_identity() == request['runtime']
_, dataset = load_cache(cfg)
assert content_hash(dataset) == request['dataset_sha256']
assert not os.environ.get('WANDB_SWEEP_ID')
FasterRCNNPipeline(cfg, work / 'run').fit()
"""

def launch_final(cfg, request, WORK, NAME, seed):
    marker = WORK / "launch_status.json"
    try:
        if marker.exists():
            metrics = verify_completed(WORK, cfg, request)
            print("Run đã hoàn tất; không train lại:", metrics["best_val"])
            return
        if not START_TRAIN:
            print("Chưa train. START_TRAIN=False")
            return
        assert not (WORK / "run").exists(), "Output dở không được ghi đè"
        run_id = uuid.uuid4().hex[:8]
        train_cfg = copy.deepcopy(cfg)
        train_cfg["train"]["run_name"] = NAME + "_" + run_id
        save_yaml(train_cfg, WORK / "train_config.yaml")
        (WORK / "worker.py").write_text(WORKER_CODE, encoding="utf8")
        command = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nnodes=1",
                   "--nproc_per_node=2", "--max_restarts=0", str(WORK / "worker.py"), str(WORK)]
        environment = dict(os.environ, WANDB_RUN_ID=run_id, WANDB_RESUME="never",
                           WANDB_PROJECT="BTL", WANDB_DIR=str(WORK),
                           WANDB_RUN_GROUP="frcnn_final_v1",
                           WANDB_TAGS=f"faster-rcnn,final,seed{seed}" + (",repeat" if seed == 42 else ""), PYTHONUNBUFFERED="1")
        for key in ("WANDB_SWEEP_ID", "WANDB_CONFIG_PATHS"):
            environment.pop(key, None)
        write_json(dict(status="running", run_id=run_id, command=command), marker)
        try:
            # Inherit stdout/stderr: Kaggle displays training progress directly.
            subprocess.run(command, cwd=REPO, env=environment, check=True)
            metrics = json.loads((WORK / "run/metrics.json").read_text())
            assert metrics["status"] == "completed"
            write_json(dict(status="completed", run_id=run_id, command=command), marker)
            verify_completed(WORK, cfg, request)
        except BaseException as exc:
            write_json(dict(status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed",
                            run_id=run_id, error_type=type(exc).__name__, command=command), marker)
            raise
    finally:
        export_output(WORK)

def run_selected_seeds(jobs):
    for index, job in enumerate(jobs, 1):
        print(f"=== Seed {job['seed']} ({index}/{len(jobs)}) ===", flush=True)
        launch_final(**job)  # Dừng khi lỗi; run trước đã được export trong finally.
    print("Hàng đợi hoàn tất (hoặc đã kiểm/skip).", flush=True)

run_selected_seeds(JOBS)


## Sau mỗi seed / sau phiên

Forge project BTL, group frcnn_final_v1. Mỗi seed có tên/run ID riêng; parent không tạo run gộp.
Tải từng cặp ZIP+SHA256 ở Output gốc về `.agents/artifacts/notebooks/frcnn_final_seeds/seed_<seed>/`.
Hai seed tạo hai cặp file. Dùng cùng input tuning cuối cho mọi phiên, không cần output seed trước trừ khi restore.
Phiên A `[7, 21]`, phiên B `[123, 2026]`; không chọn cùng seed ở hai phiên đồng thời.
Nếu chọn 42, run có tag repeat và request.repeat_of; không tự thay ahgf5kge hoặc cộng thành seed thứ sáu.
Sau đủ bốn seed mới, review rồi tổng hợp với seed42 gốc, mean và sample std ddof=1.
Test chỉ report, không chọn/bỏ seed theo điểm. Hard session kill không bảo đảm finally export;
giữ Save Version outputs. Không tự upload model Artifact lên Forge.
